# Cancer Risk Assessment — Group 28

This is the cleaned notebook version of the supplied group coursework. It implements exploratory data analysis, a Random Forest classifier and a pruned Decision Tree. The group report also describes multiclass logistic regression and Bayesian models, but executable scripts for those methods were not included in the supplied archive.

**Data notice:** the raw dataset is not committed. Obtain an authorised copy and place it at data/cancer_risk_dataset.csv. Check source, licence and de-identification before use. The model is an educational exercise, not a clinical diagnostic tool.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier, plot_tree

DATA_PATH = Path('data/cancer_risk_dataset.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Place an authorised dataset at {DATA_PATH}')

df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df.head())
print(df['Level'].value_counts(dropna=False))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Level', order=df['Level'].value_counts().index)
plt.title('Distribution of risk-level labels')
plt.tight_layout()
plt.show()

target = 'Level'
drop_cols = [c for c in ['Patient Id', 'index', target] if c in df.columns]
X = df.drop(columns=drop_cols).copy()
y = LabelEncoder().fit_transform(df[target].astype(str))
X = pd.get_dummies(X, drop_first=False)
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True)).fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
print('Weighted F1:', f1_score(y_test, pred, average='weighted'))
print(classification_report(y_test, pred, zero_division=0))
print('Confusion matrix:\n', confusion_matrix(y_test, pred))

importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance.head(15))
importance.head(15).sort_values().plot.barh(figsize=(8, 5), title='Random Forest feature importance')
plt.tight_layout()
plt.show()

# Pruned Decision Tree: choose alpha using cross-validation, then tune further.
base_tree = DecisionTreeClassifier(random_state=42)
path = base_tree.cost_complexity_pruning_path(X_train, y_train)
alpha_grid = np.unique(path.ccp_alphas)
alpha_grid = alpha_grid[::max(1, len(alpha_grid)//50)]
alpha_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {'ccp_alpha': alpha_grid}, cv=5, scoring='accuracy', n_jobs=-1
)
alpha_search.fit(X_train, y_train)
tree = DecisionTreeClassifier(ccp_alpha=alpha_search.best_params_['ccp_alpha'], random_state=42)
param_dist = {
    'criterion': ['gini', 'entropy'],
    'min_samples_split': list(range(2, 31)),
    'min_samples_leaf': list(range(1, 21)),
    'max_features': ['sqrt', 'log2', None]
}
tree_search = RandomizedSearchCV(
    tree, param_distributions=param_dist, n_iter=30, cv=5,
    scoring='accuracy', random_state=42, n_jobs=-1
)
tree_search.fit(X_train, y_train)
print('Decision Tree best CV accuracy:', tree_search.best_score_)
print('Decision Tree test classification report:')
print(classification_report(y_test, tree_search.best_estimator_.predict(X_test), zero_division=0))


## Interpretation and audit checklist

The original coursework notebook reported a weighted Random Forest F1 of 1.00 and very high Decision Tree cross-validation accuracy. Such near-perfect results on a small educational dataset require auditing for target construction, feature leakage, duplicates, class balance and split design. They are not evidence of clinical validity. Feature importance is predictive association, not causal explanation.

This is a cleaned, documented notebook based on the supplied workflow. Its preprocessing and stratified split differ from the original notebook, so its output should not be expected to reproduce the original score exactly.